# Self-test: is your teaching environment ready?

This notebook checks that the environment for the teaching materials works: the Python
kernel, the installed packages, the example data, the persistent `work/` folder, the
terminal, registering a kernel, and Git with the JupyterLab Git panel.

**How to run it**

1. Open a terminal (**File → New → Terminal**) and paste this line, then press Enter:

   ```bash
   mkdir -p ~/work/.self-test && { echo "$GIT_CONFIG_GLOBAL"; python --version; } > ~/work/.self-test/terminal.txt
   ```

2. The lab takes your Git name and email from your Jupyter4NFDI account when it starts.
   Your email goes into every commit, and anyone can read it in a repository you publish.
   To use another name or email, or if the Git check below finds none, set them in the
   same terminal (once; it is kept in `work/`, and a value you set is never replaced):

   ```bash
   git config --global user.name "Your Name"
   git config --global user.email "you@example.org"
   ```

3. Come back to this notebook and choose **Run → Run All Cells**.

Every check prints `PASS` or `FAIL`. A failed check also prints **What to do**. The last
cell sums up. When everything passes, stop the lab (**File → Hub Control Panel → Stop
My Server**), start it again with the same settings, and run this notebook once more:
the persistence check then confirms that your files survived the restart.

The notebook changes nothing outside `work/.self-test/`, a hidden folder it keeps for
the persistence check. You can delete that folder when you are done.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from importlib import metadata
from pathlib import Path

HOME = Path.home()
WORK = HOME / "work"            # the folder that is kept between sessions
STATE = WORK / ".self-test"     # where this notebook keeps its own files
REPO = Path.cwd()               # this notebook sits at the root of the teaching materials
results = {}


def report(name, ok, detail, fix):
    """Print one check result; on failure, print what to do."""
    results[name] = ok
    print(("PASS" if ok else "FAIL") + f"  {name}: {detail}")
    if not ok:
        print(f"      What to do: {fix}")


def run(*cmd, timeout=60):
    """Run a command and return (exit code, combined output)."""
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return p.returncode, (p.stdout + p.stderr).strip()
    except FileNotFoundError:
        return 127, f"{cmd[0]}: command not found"
    except subprocess.TimeoutExpired:
        return 124, f"{' '.join(cmd)}: no answer after {timeout} s"


print("Checks start:", time.strftime("%Y-%m-%d %H:%M:%S"))

## 1. Python kernel

**Expected:** `PASS  Python kernel: Python 3.13.x at /srv/conda/envs/notebook/bin/python3.13`

In [ ]:
prefix = os.environ.get("NB_PYTHON_PREFIX", "/srv/conda/envs/notebook")
ok = sys.version_info[:2] == (3, 13) and Path(sys.prefix) == Path(prefix)
report(
    "Python kernel", ok, f"Python {sys.version.split()[0]} at {sys.executable}",
    "This notebook runs on another kernel. Choose Kernel → Change Kernel… → "
    "'Python 3 (ipykernel)' and run all cells again. If that kernel is missing, the lab "
    "was not started from the teaching materials: stop it (File → Hub Control Panel → "
    "Stop My Server) and start it again as the preparation instructions describe.",
)

## 2. Packages

Every package in `requirements.txt` is installed in exactly the pinned version and
imports without error.

**Expected:** a table with `ok` in every row, then `PASS  Packages: 13 packages as pinned`

In [ ]:
req = REPO / "requirements.txt"
pins = {}
if req.exists():
    for line in req.read_text().splitlines():
        line = line.split("#")[0].strip()
        if "==" in line:
            name, version = (part.strip() for part in line.split("=="))
            pins[name] = version

# Distribution name -> module name, where they differ
modules = {"jupyterlab-git": "jupyterlab_git", "jupyter-collaboration": "jupyter_collaboration",
           "jupyter-resource-usage": "jupyter_resource_usage"}
problems = []
for name, wanted in pins.items():
    try:
        installed = metadata.version(name)
        __import__(modules.get(name, name))
        state = "ok" if installed == wanted else "WRONG VERSION"
    except metadata.PackageNotFoundError:
        installed, state = "-", "MISSING"
    except Exception as error:  # installed but broken
        installed, state = metadata.version(name), f"IMPORT FAILS ({type(error).__name__})"
    if state != "ok":
        problems.append(name)
    print(f"   {name:24} pinned {wanted:10} installed {installed:10} {state}")

report(
    "Packages", bool(pins) and not problems,
    f"{len(pins)} packages as pinned" if pins and not problems
    else (f"problem with {', '.join(problems)}" if pins else "requirements.txt not found next to this notebook"),
    "The packages come with the lab image, so a restart restores them: stop the lab "
    "(File → Hub Control Panel → Stop My Server) and start it again with the repository "
    "and Git Ref from the preparation instructions. Do not install or upgrade packages "
    "with pip in this environment. If the check still fails, send this cell's output to the tutor.",
)

## 3. Example data

Reads the example dataset (*E. coli* growth at 37 °C, 1 % glucose), checks that it is
unchanged, and runs it once through pandas, SciPy, xarray and Matplotlib.

**Expected:** `PASS  Example data: 289 readings, 8 wells, last mean OD 0.580, fastest growth 0.69 per hour at 0.9 h`,
followed by a plot of a growth curve that rises from about 0.14 to about 0.58.

In [ ]:
sample = REPO / "data" / "ecoli-growth" / "ecoli-growth-37c-1pct.csv"
expected_sha256 = "b7d99a36eed7cbfcd567a3ca4e9950f92a70a808176c92c9c7a1bb64e4d82324"
fix_data = ("The example data is missing or was changed. The original comes back with every "
            "lab start: stop the lab (File → Hub Control Panel → Stop My Server) and start it "
            "again. To work with the data, copy it into work/ and change the copy.")
try:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import xarray as xr
    from scipy.signal import savgol_filter

    digest = hashlib.sha256(sample.read_bytes()).hexdigest()
    df = pd.read_csv(sample)
    wells = [c for c in df.columns if len(c) == 2 and c[0] in "ABCDEFGH"]
    hours = df["tcultivo (min)"] / 60
    od = xr.DataArray(df[wells].to_numpy(), dims=("time_h", "well"),
                      coords={"time_h": hours.to_numpy(), "well": wells})
    mean_od = od.mean("well")
    rate = np.gradient(savgol_filter(np.log(mean_od.values), 11, 2), hours)
    fastest = rate.argmax()

    fig, ax = plt.subplots(figsize=(6, 3))
    mean_od.plot(ax=ax)
    ax.set(xlabel="time (h)", ylabel="mean optical density", title="E. coli, 37 °C, 1 % glucose")
    plt.show()

    ok = digest == expected_sha256 and len(df) == 289 and len(wells) == 8
    report("Example data", ok,
           f"{len(df)} readings, {len(wells)} wells, last mean OD {float(mean_od[-1]):.3f}, "
           f"fastest growth {rate[fastest]:.2f} per hour at {hours[fastest]:.1f} h"
           + ("" if digest == expected_sha256 else " — the file differs from the original"),
           fix_data)
except FileNotFoundError:
    report("Example data", False, f"{sample.relative_to(REPO)} not found", fix_data)
except ImportError as error:
    report("Example data", False, f"cannot import {error.name}",
           "A package is missing. Fix check 2 (Packages) first.")

## 4. Persistent `work/` folder

Only the `work/` folder in your home folder is kept when the lab stops. Everything else,
including this notebook, is reset at the next start. This check confirms that `work/` is
persistent storage and that you can write to it.

**Expected on the first run:** `PASS  Persistent work folder: ... first run: restart the lab and run this notebook again`

**Expected after a restart:** `PASS  Persistent work folder: ... found the note from <date and time of the first run>`

In [ ]:
stamp = STATE / "persistence.txt"
fix_mount = ("The lab was started without persistent storage, so your work is lost when it "
             "stops. Stop the lab (File → Hub Control Panel → Stop My Server) and start it "
             "again with 'Mount user data' switched on and the path /home/jovyan/work.")
mounted = os.path.ismount(WORK)
try:
    previous = stamp.read_text().strip() if stamp.exists() else None
    STATE.mkdir(parents=True, exist_ok=True)
    now = time.strftime("%Y-%m-%d %H:%M:%S")
    stamp.write_text(now + "\n")
    writable = stamp.read_text().strip() == now
except OSError as error:
    previous, writable = None, False
    print(f"   cannot write to {STATE}: {error}")

if not mounted:
    report("Persistent work folder", False, f"{WORK} is not persistent storage", fix_mount)
elif not writable:
    report("Persistent work folder", False, f"cannot write to {WORK}",
           "Your storage may be full: delete files you no longer need in work/ and run this cell again. "
           "If it still fails, send this cell's output to the tutor.")
elif previous:
    report("Persistent work folder", True,
           f"{WORK} is persistent; found the note from {previous}", "")
else:
    report("Persistent work folder", True,
           f"{WORK} is persistent; first run: restart the lab and run this notebook again", "")

## 5. Terminal

This check reads the file that the line from step 1 at the top wrote in the terminal. It
deletes the file afterwards, so every run needs that line again.

**Expected:** `PASS  Terminal: the terminal runs Python 3.13.x with Git settings in /home/jovyan/work/.gitconfig`

In [ ]:
note = STATE / "terminal.txt"
fix_terminal = ("Open File → New → Terminal, paste the line from step 1 at the top of this "
                "notebook, press Enter, and run this cell again. If there is no Terminal in "
                "the File → New menu, stop and restart the lab; if it is still missing, tell the tutor.")
if not note.exists():
    report("Terminal", False, "no answer from the terminal yet", fix_terminal)
else:
    lines = (note.read_text().splitlines() + ["", ""])[:2]
    note.unlink()
    gitconfig, version = lines[0].strip(), lines[1].strip()
    ok = gitconfig == str(WORK / ".gitconfig") and version.startswith("Python 3.13")
    report("Terminal", ok,
           f"the terminal runs {version or 'no Python'} with Git settings in {gitconfig or '(not set)'}",
           "The terminal does not run the environment of the teaching materials. Close all "
           "terminals (Running Terminals and Kernels, left sidebar), stop and restart the lab, "
           "and repeat step 1.")

## 6. Registering a kernel

Later you will build your own environments and register them as Jupyter kernels. This
check registers a temporary kernel, confirms that Jupyter lists it, and removes it again.

**Expected:** `PASS  Kernel registration: registered and removed 'self-test'; the default kernel is python3`

In [ ]:
name = "self-test"
code_install, out_install = run(sys.executable, "-m", "ipykernel", "install", "--user",
                                "--name", name, "--display-name", "Python (self-test)")
code_list, out_list = run(sys.executable, "-m", "jupyter", "kernelspec", "list", "--json")
try:
    specs = json.loads(out_list[out_list.find("{"):])["kernelspecs"]
except ValueError:
    specs = {}
registered = code_install == 0 and name in specs
run(sys.executable, "-m", "jupyter", "kernelspec", "remove", "-y", name)
if not registered:
    print("   " + (out_install or out_list).replace("\n", "\n   "))
report("Kernel registration", registered and "python3" in specs,
       f"registered and removed '{name}'; the default kernel is python3" if registered
       else "Jupyter did not list the new kernel",
       "Stop and restart the lab, then run this cell again. If it still fails, send this "
       "cell's output to the tutor.")

## 7. Git and the Git panel

Checks the Git command, your Git name and email, a test commit, and that the Git panel
is installed with its settings for the course: one-step commits (*simple staging*).

**Expected:** `PASS  Git: git version 2.x, commits as <your name> <your email>, test commit made`
and `PASS  Git panel: jupyterlab-git installed, simple staging on`

Then look at the left sidebar of JupyterLab: it has a Git tab (the icon with a branch
symbol). Opened in a folder that is not a repository, it offers **Initialize a Repository**.

In [ ]:
fix_identity = ("The lab could not take your Git name and email from your Jupyter4NFDI account.\n"
                "      Set them once in a terminal (File → New → Terminal):\n"
                '        git config --global user.name "Your Name"\n'
                '        git config --global user.email "you@example.org"\n'
                "      then run this cell again.")
_, git_version = run("git", "--version")
_, user = run("git", "config", "--global", "user.name")
_, email = run("git", "config", "--global", "user.email")
gitconfig = os.environ.get("GIT_CONFIG_GLOBAL", "")

if not git_version.startswith("git version"):
    report("Git", False, git_version, "Git is missing from the lab image. Tell the tutor.")
elif gitconfig != str(WORK / ".gitconfig"):
    report("Git", False, f"Git settings are kept in {gitconfig or HOME / '.gitconfig'}, not in work/",
           "The lab was not started from the teaching materials. Start it again as the "
           "preparation instructions describe.")
elif not (user and email):
    report("Git", False, "no Git name or email set", fix_identity)
else:
    repo = STATE / "git-check"
    shutil.rmtree(repo, ignore_errors=True)
    repo.mkdir(parents=True)
    (repo / "notes.md").write_text("Self-test\n")
    steps = [run("git", "-C", str(repo), "init", "-q"),
             run("git", "-C", str(repo), "add", "notes.md"),
             run("git", "-C", str(repo), "commit", "-q", "-m", "Self-test commit", timeout=30)]
    committed = all(code == 0 for code, _ in steps)
    shutil.rmtree(repo, ignore_errors=True)
    if not committed:
        print("   " + "\n   ".join(out for code, out in steps if code))
    report("Git", committed,
           f"{git_version}, commits as {user} <{email}>, "
           + ("test commit made" if committed else "test commit failed"),
           "Check your name and email with 'git config --global --list' in a terminal and "
           "set them again if needed (see the top of this notebook). If the commit still "
           "fails, send this cell's output to the tutor.")

overrides = Path(os.environ.get("NB_PYTHON_PREFIX", sys.prefix)) / "share/jupyter/lab/settings/overrides.json"
try:
    panel = json.loads(overrides.read_text()).get("@jupyterlab/git:plugin", {})
except (OSError, ValueError):
    panel = {}
try:
    panel_version = metadata.version("jupyterlab-git")
except metadata.PackageNotFoundError:
    panel_version = None
report("Git panel", bool(panel_version) and panel.get("simpleStaging") is True,
       f"jupyterlab-git {panel_version or 'not installed'}, simple staging "
       + ("on" if panel.get("simpleStaging") else "off"),
       "The lab image lacks the Git panel or its settings. Start the lab again from the "
       "teaching materials as the preparation instructions describe; if it persists, tell the tutor.")

## Summary

**Expected:** `All 8 checks passed. Your environment is ready.`

In [ ]:
failed = [name for name, ok in results.items() if not ok]
if failed:
    print(f"{len(failed)} of {len(results)} checks failed: {', '.join(failed)}.")
    print("Follow 'What to do' under each failed check, then run all cells again.")
else:
    print(f"All {len(results)} checks passed. Your environment is ready.")